In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"

In [2]:
import gc
import re
import sys
import json
import time
import math
import random
import unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup
)
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_WINDOWS = sys.platform.startswith("win")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", torch.cuda.get_device_properties(0).total_memory / 1e9)

Device: cuda
GPU: Tesla T4
VRAM GB: 15.936192512


In [3]:
TASK1_TRAIN = r"C:/Users/VGST/Desktop/SciHigh-2026/Task 1/MixSub-SciHigh_train_FIRE.csv"
TASK1_VAL = r"C:/Users/VGST/Desktop/SciHigh-2026/Task 1/MixSub-SciHigh_val_FIRE.csv"
TASK1_TEST = r"C:/Users/VGST/Desktop/SciHigh-2026/Task 1/MixSub-SciHigh_test_FIRE(Masked).csv"

TASK2_TRAIN = r"C:/Users/VGST/Desktop/SciHigh-2026/Task 2/SpringerSSAT-Train-SciHigh.csv"
TASK2_VAL = r"C:/Users/VGST/Desktop/SciHigh-2026/Task 2/SpringerSSAT-Val-SciHigh.csv"
TASK2_TEST = r"C:/Users/VGST/Desktop/SciHigh-2026/Task 2/SpringerSSAT-Test-SciHigh(Masked).csv"

TASK3_TRAIN = r"C:/Users/VGST/Desktop/SciHigh-2026/Task 3/SpringerSSAT-Tiny-Multilingual-train.csv"
TASK3_VAL = r"C:/Users/VGST/Desktop/SciHigh-2026/Task 3/SpringerSSAT-Tiny-Multilingual-val.csv"
TASK3_TEST = r"C:/Users/VGST/Desktop/SciHigh-2026/Task 3/SpringerSSAT-Tiny-Multilingual-test(Masked).csv"

OUTPUT_DIR = os.path.normpath(r"C:\Users\VGST\Desktop\SciHigh-2026\outputs")
CKPT_DIR = os.path.normpath(os.path.join(OUTPUT_DIR, "checkpoints"))
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

In [4]:
SMART_CHAR_MAP = {
    "\u2018": "'", "\u2019": "'",
    "\u201c": '"', "\u201d": '"',
    "\u2013": "-", "\u2014": "-",
    "\u2026": "..."
}

def normalize_unicode(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    for bad, good in SMART_CHAR_MAP.items():
        text = text.replace(bad, good)
    return text

def clean_whitespace(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_abstract(text):
    text = normalize_unicode(text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\[\d+(,\s*\d+)*\]", "", text)
    text = clean_whitespace(text)
    return text

def clean_target(text):
    text = normalize_unicode(text)
    text = clean_whitespace(text)
    return text

def read_csv_robust(path):
    encodings_to_try = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_err = None
    for enc in encodings_to_try:
        try:
            df = pd.read_csv(path, encoding=enc)
            if enc != "utf-8":
                print(f"Loaded {path} using fallback encoding: {enc}")
            return df
        except UnicodeDecodeError as e:
            last_err = e
            continue
    raise last_err

def load_and_clean_csv(path, text_cols):
    df = read_csv_robust(path)
    df = df.drop_duplicates()
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("")
            if "abstract" in col.lower():
                df[col] = df[col].apply(clean_abstract)
            else:
                df[col] = df[col].apply(clean_target)
    for col in text_cols:
        if col in df.columns:
            df = df[df[col].str.len() > 0]
    df = df.reset_index(drop=True)
    return df

def resolve_column(df, candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    raise KeyError(
        f"None of {candidates} found in dataframe. Available columns: {df.columns.tolist()}"
    )

In [5]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()

def gpu_mem_report(tag=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"[{tag}] allocated={alloc:.2f}GB reserved={reserved:.2f}GB")

def resolve_checkpoint_path(save_dir):
    save_dir = os.path.normpath(save_dir)
    best_path = os.path.normpath(os.path.join(save_dir, "best_checkpoint"))
    last_path = os.path.normpath(os.path.join(save_dir, "last_checkpoint"))

    if os.path.isdir(best_path) and os.path.exists(os.path.join(best_path, "config.json")):
        return best_path
    if os.path.isdir(last_path) and os.path.exists(os.path.join(last_path, "config.json")):
        print(f"best_checkpoint not found in {save_dir}, using last_checkpoint instead")
        return last_path

    raise FileNotFoundError(
        f"No usable checkpoint found in {save_dir}. "
        f"Checked {best_path} and {last_path}. "
        f"Training for this model may not have completed a single successful epoch."
    )

def load_with_retry(load_fn, *args, max_retries=5, initial_wait=5, **kwargs):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return load_fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            wait = initial_wait * (2 ** (attempt - 1))
            print(f"Download/load attempt {attempt}/{max_retries} failed: {type(e).__name__}: {e}")
            if attempt < max_retries:
                print(f"Retrying in {wait} seconds...")
                time.sleep(wait)
    raise RuntimeError(f"Failed after {max_retries} attempts. Last error: {last_err}") from last_err

In [6]:
class Seq2SeqDataset(Dataset):
    def __init__(self, sources, targets, tokenizer, max_source_len=512, max_target_len=128):
        self.tokenizer = tokenizer
        self.max_source_len = max_source_len
        self.max_target_len = max_target_len

        sources = list(sources)
        targets = list(targets)

        source_enc = tokenizer(
            sources,
            max_length=max_source_len,
            truncation=True
        )
        target_enc = tokenizer(
            text_target=targets,
            max_length=max_target_len,
            truncation=True
        )

        self.input_ids = source_enc["input_ids"]
        self.attention_mask = source_enc["attention_mask"]
        self.labels = target_enc["input_ids"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

    def get_lengths(self):
        return [len(ids) for ids in self.input_ids]


class BucketedLengthSampler(Sampler):
    def __init__(self, lengths, batch_size, shuffle=True, bucket_size_multiplier=50):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.bucket_size = batch_size * bucket_size_multiplier

    def __iter__(self):
        indices = list(range(len(self.lengths)))
        if self.shuffle:
            random.shuffle(indices)

        buckets = [
            indices[i:i + self.bucket_size]
            for i in range(0, len(indices), self.bucket_size)
        ]

        batches = []
        for bucket in buckets:
            bucket_sorted = sorted(bucket, key=lambda idx: self.lengths[idx])
            bucket_batches = [
                bucket_sorted[i:i + self.batch_size]
                for i in range(0, len(bucket_sorted), self.batch_size)
            ]
            batches.extend(bucket_batches)

        if self.shuffle:
            random.shuffle(batches)

        for batch in batches:
            for idx in batch:
                yield idx

    def __len__(self):
        return len(self.lengths)

In [7]:
import evaluate as hf_evaluate

rouge_metric = hf_evaluate.load("rouge")
meteor_metric = hf_evaluate.load("meteor")
bertscore_metric = hf_evaluate.load("bertscore")
bleu_metric = hf_evaluate.load("sacrebleu")
chrf_metric = hf_evaluate.load("chrf")

def compute_all_metrics(preds, refs, lang="en"):
    preds = [p.strip() if p.strip() else "empty" for p in preds]
    refs = [r.strip() if r.strip() else "empty" for r in refs]

    rouge_res = rouge_metric.compute(predictions=preds, references=refs)
    meteor_res = meteor_metric.compute(predictions=preds, references=refs)
    bertscore_res = bertscore_metric.compute(predictions=preds, references=refs, lang=lang)
    bleu_res = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    chrf_res = chrf_metric.compute(predictions=preds, references=[[r] for r in refs])

    avg_len = float(np.mean([len(p.split()) for p in preds]))

    return {
        "rouge1": rouge_res["rouge1"],
        "rouge2": rouge_res["rouge2"],
        "rougeL": rouge_res["rougeL"],
        "meteor": meteor_res["meteor"],
        "bertscore_f1": float(np.mean(bertscore_res["f1"])),
        "bleu": bleu_res["score"],
        "chrf": chrf_res["score"],
        "avg_length": avg_len
    }

_comet_model_cache = {"model": None, "unavailable": False}

def compute_comet(sources, preds, refs):
    if _comet_model_cache["unavailable"]:
        print("COMET unavailable in this environment, returning None")
        return None

    if _comet_model_cache["model"] is None:
        try:
            from comet import download_model, load_from_checkpoint
            model_path = download_model("Unbabel/wmt22-comet-da")
            _comet_model_cache["model"] = load_from_checkpoint(model_path)
        except Exception as e:
            print(f"COMET model download/load failed: {e}")
            print("Falling back to skipping COMET score.")
            _comet_model_cache["unavailable"] = True
            return None

    try:
        data = [{"src": s, "mt": p, "ref": r} for s, p, r in zip(sources, preds, refs)]
        output = _comet_model_cache["model"].predict(data, batch_size=8, gpus=1 if torch.cuda.is_available() else 0)
        return float(output.system_score)
    except Exception as e:
        print(f"COMET scoring failed: {e}")
        return None

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\VGST\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\VGST\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\VGST\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [8]:
def train_seq2seq_model(
    model_name,
    train_dataset,
    val_dataset,
    tokenizer,
    save_dir,
    max_epochs=4,
    base_batch_size=4,
    grad_accum_steps=8,
    lr=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    label_smoothing=0.1,
    patience=2,
    min_batch_size=1,
    oom_tolerance=3
):
    save_dir = os.path.normpath(save_dir)
    os.makedirs(save_dir, exist_ok=True)
    resume_path = os.path.normpath(os.path.join(save_dir, "last_checkpoint"))

    if os.path.isdir(resume_path) and os.path.exists(os.path.join(resume_path, "config.json")):
        model = AutoModelForSeq2SeqLM.from_pretrained(resume_path, local_files_only=True)
        print(f"Resumed model from {resume_path}")
    else:
        model = load_with_retry(AutoModelForSeq2SeqLM.from_pretrained, model_name)

    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model.to(DEVICE)

    gpu_mem_report(f"before starting training for {model_name}")
    if torch.cuda.is_available():
        allocated_gb = torch.cuda.memory_allocated() / 1e9
        if allocated_gb > 3.0:
            print(f"WARNING: {allocated_gb:.2f}GB already allocated before this model even loads. A previous model was likely not freed.")

    collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

    num_workers = 0 if IS_WINDOWS else 2
    use_persistent = num_workers > 0

    def make_loader(bs):
        lengths = train_dataset.get_lengths()
        sampler = BucketedLengthSampler(lengths, bs, shuffle=True)
        return DataLoader(
            train_dataset,
            batch_size=bs,
            sampler=sampler,
            collate_fn=collator,
            num_workers=num_workers,
            pin_memory=True,
            persistent_workers=use_persistent
        )

    def make_val_loader(bs):
        return DataLoader(
            val_dataset,
            batch_size=bs,
            shuffle=False,
            collate_fn=collator,
            num_workers=num_workers,
            pin_memory=True,
            persistent_workers=use_persistent
        )

    current_bs = base_batch_size
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing, ignore_index=-100)
    best_val_loss = float("inf")
    epochs_no_improve = 0

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    approx_steps_per_epoch = max(1, len(train_dataset) // (current_bs * grad_accum_steps))
    total_steps = max(1, approx_steps_per_epoch * max_epochs)
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler = GradScaler()

    epoch = 0
    while epoch < max_epochs:
        train_loader = make_loader(current_bs)

        model.train()
        optimizer.zero_grad()
        running_loss = 0.0
        step_count = 0
        consecutive_oom = 0
        total_oom_this_epoch = 0
        epoch_failed = False

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs} bs={current_bs} train", leave=True)
        step_start = time.time()

        for step, batch in enumerate(pbar):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            try:
                with autocast():
                    outputs = model(
                        input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"],
                        labels=batch["labels"]
                    )
                    logits = outputs.logits
                    labels = batch["labels"]
                    loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
                    loss = loss / grad_accum_steps

                scaler.scale(loss).backward()

                if (step + 1) % grad_accum_steps == 0:
                    try:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                        scheduler.step()
                    except torch.cuda.OutOfMemoryError:
                        optimizer.zero_grad()
                        clear_memory()
                        scaler = GradScaler()
                        consecutive_oom += 1
                        total_oom_this_epoch += 1
                        print(f"OOM during optimizer step at step {step}, scaler reset, consecutive_oom={consecutive_oom}")
                        if consecutive_oom >= oom_tolerance or total_oom_this_epoch >= oom_tolerance * 3:
                            if current_bs <= min_batch_size:
                                raise RuntimeError(f"OOM persists even at minimum batch size {min_batch_size}.")
                            current_bs = max(min_batch_size, current_bs // 2)
                            print(f"Reducing batch size to {current_bs} and restarting this epoch")
                            epoch_failed = True
                            break
                        continue
                    optimizer.zero_grad()

                running_loss += loss.item() * grad_accum_steps
                step_count += 1
                consecutive_oom = 0

                elapsed = time.time() - step_start
                it_per_sec = (step + 1) / max(elapsed, 1e-6)
                pbar.set_postfix(loss=round(running_loss / step_count, 4), it_per_sec=round(it_per_sec, 3), bs=current_bs)

            except torch.cuda.OutOfMemoryError:
                optimizer.zero_grad()
                clear_memory()
                consecutive_oom += 1
                total_oom_this_epoch += 1
                print(f"OOM at step {step}, consecutive_oom={consecutive_oom}, total_this_epoch={total_oom_this_epoch}, current batch size={current_bs}")
                if consecutive_oom >= oom_tolerance or total_oom_this_epoch >= oom_tolerance * 3:
                    if current_bs <= min_batch_size:
                        raise RuntimeError(f"OOM persists even at minimum batch size {min_batch_size}. Switch to a smaller model or shorter sequences.")
                    current_bs = max(min_batch_size, current_bs // 2)
                    print(f"Reducing batch size to {current_bs} and restarting this epoch")
                    epoch_failed = True
                    break

            if step % 50 == 0:
                clear_memory()

        del train_loader
        clear_memory()

        if epoch_failed:
            scaler = GradScaler()
            continue

        avg_train_loss = running_loss / max(1, step_count)
        print(f"Epoch {epoch+1} train_loss={avg_train_loss:.4f} final_batch_size={current_bs}")

        val_loader = make_val_loader(current_bs)
        model.eval()
        val_loss = 0.0
        val_steps = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{max_epochs} val", leave=True):
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                try:
                    with autocast():
                        outputs = model(
                            input_ids=batch["input_ids"],
                            attention_mask=batch["attention_mask"],
                            labels=batch["labels"]
                        )
                    val_loss += outputs.loss.item()
                    val_steps += 1
                except torch.cuda.OutOfMemoryError:
                    clear_memory()
                    continue

        del val_loader
        clear_memory()

        avg_val_loss = val_loss / max(1, val_steps)
        print(f"Epoch {epoch+1} val_loss={avg_val_loss:.4f}")

        model.save_pretrained(resume_path)
        tokenizer.save_pretrained(resume_path)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            best_path = os.path.normpath(os.path.join(save_dir, "best_checkpoint"))
            model.save_pretrained(best_path)
            tokenizer.save_pretrained(best_path)
            print(f"New best checkpoint saved at {best_path}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break

        clear_memory()
        epoch += 1

    del optimizer, scheduler, scaler
    clear_memory()

    best_path = resolve_checkpoint_path(save_dir)

    del model
    clear_memory()

    final_model = AutoModelForSeq2SeqLM.from_pretrained(best_path, local_files_only=True)
    final_model.config.use_cache = True
    final_model.to(DEVICE)
    gpu_mem_report(f"after loading best checkpoint from {best_path}")
    return final_model

In [9]:
def generate_predictions(model, tokenizer, texts, max_source_len=512, max_target_len=128,
                          num_beams=4, length_penalty=1.0, repetition_penalty=1.2, batch_size=8,
                          forced_bos_token_id=None):
    model.eval()
    predictions = []
    i = 0
    bs = batch_size
    while i < len(texts):
        batch_texts = texts[i:i+bs]
        try:
            inputs = tokenizer(
                batch_texts,
                max_length=max_source_len,
                truncation=True,
                padding=True,
                return_tensors="pt"
            ).to(DEVICE)

            gen_kwargs = dict(
                max_length=max_target_len,
                num_beams=num_beams,
                length_penalty=length_penalty,
                repetition_penalty=repetition_penalty,
                early_stopping=True
            )
            if forced_bos_token_id is not None:
                gen_kwargs["forced_bos_token_id"] = forced_bos_token_id

            with torch.no_grad(), autocast():
                output_ids = model.generate(**inputs, **gen_kwargs)

            decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
            predictions.extend(decoded)
            i += bs
            clear_memory()

        except torch.cuda.OutOfMemoryError:
            clear_memory()
            bs = max(1, bs // 2)
            print(f"OOM during generation, reducing batch size to {bs}")

    return predictions

def load_model_and_tokenizer(path):
    path = os.path.normpath(path)
    if not os.path.isdir(path):
        raise FileNotFoundError(f"Checkpoint directory does not exist: {path}")
    if not os.path.exists(os.path.join(path, "config.json")):
        raise FileNotFoundError(f"Checkpoint directory exists but has no config.json: {path}")

    tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(path, local_files_only=True)
    model.config.use_cache = True
    model.to(DEVICE)
    return model, tokenizer

def generate_from_checkpoint(path, texts, **gen_kwargs):
    model, tokenizer = load_model_and_tokenizer(path)
    preds = generate_predictions(model, tokenizer, texts, **gen_kwargs)
    del model, tokenizer
    clear_memory()
    return preds

Task 1

In [10]:
t1_train = load_and_clean_csv(TASK1_TRAIN, ["Abstract", "Highlights"])
t1_val = load_and_clean_csv(TASK1_VAL, ["Abstract", "Highlights"])
t1_test = load_and_clean_csv(TASK1_TEST, ["Abstract"])

print(t1_train.shape, t1_val.shape, t1_test.shape)
print(t1_train.columns.tolist())

(10000, 3) (1985, 3) (1840, 2)
['Filename', 'Abstract', 'Highlights']


In [11]:
T1_MODEL_A_NAME = "facebook/bart-base"
T1_MODEL_A_DIR = os.path.join(CKPT_DIR, "task1_model_a")

t1_tok_a = AutoTokenizer.from_pretrained(T1_MODEL_A_NAME)
t1_train_ds_a = Seq2SeqDataset(t1_train["Abstract"], t1_train["Highlights"], t1_tok_a, 512, 128)
t1_val_ds_a = Seq2SeqDataset(t1_val["Abstract"], t1_val["Highlights"], t1_tok_a, 512, 128)

t1_model_a = train_seq2seq_model(
    T1_MODEL_A_NAME,
    t1_train_ds_a,
    t1_val_ds_a,
    t1_tok_a,
    save_dir=T1_MODEL_A_DIR,
    max_epochs=4,
    base_batch_size=4,
    grad_accum_steps=8,
    lr=3e-5
)

t1_val_preds_a = generate_predictions(t1_model_a, t1_tok_a, t1_val["Abstract"].tolist())
t1_metrics_a = compute_all_metrics(t1_val_preds_a, t1_val["Highlights"].tolist())
print("Task1 Model A metrics:", t1_metrics_a)

del t1_model_a, t1_train_ds_a, t1_val_ds_a
clear_memory()
gpu_mem_report("after freeing task1 model a")

c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\models\bart\configuration_bart.py:176: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


Resumed model from C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task1_model_a\last_checkpoint


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


[before starting training for facebook/bart-base] allocated=1.63GB reserved=1.64GB


C:\Users\VGST\AppData\Local\Temp\ipykernel_17028\29908453.py:75: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch 1/4 bs=4 train:   0%|          | 0/2500 [00:00<?, ?it/s]

... (19 outputs left out)

c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\models\bart\configuration_bart.py:176: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


[after loading best checkpoint from C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task1_model_a\best_checkpoint] allocated=4.97GB reserved=5.09GB


C:\Users\VGST\AppData\Local\Temp\ipykernel_17028\470968428.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\utils.py:1493: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Task1 Model A metrics: {'rouge1': np.float64(0.09133839797531185), 'rouge2': np.float64(0.028111620967696357), 'rougeL': np.float64(0.06370725769380521), 'meteor': np.float64(0.0796406758924447), 'bertscore_f1': 0.811976664853036, 'bleu': 1.1304268619401365, 'chrf': 12.961839254149195, 'avg_length': 18.385894206549118}
[after freeing task1 model a] allocated=1.09GB reserved=1.33GB


In [12]:
T1_MODEL_B_NAME = "google/pegasus-xsum"
T1_MODEL_B_DIR = os.path.join(CKPT_DIR, "task1_model_b")

t1_tok_b = AutoTokenizer.from_pretrained(T1_MODEL_B_NAME)
t1_train_ds_b = Seq2SeqDataset(t1_train["Abstract"], t1_train["Highlights"], t1_tok_b, 512, 128)
t1_val_ds_b = Seq2SeqDataset(t1_val["Abstract"], t1_val["Highlights"], t1_tok_b, 512, 128)

t1_model_b = train_seq2seq_model(
    T1_MODEL_B_NAME,
    t1_train_ds_b,
    t1_val_ds_b,
    t1_tok_b,
    save_dir=T1_MODEL_B_DIR,
    max_epochs=4,
    base_batch_size=4,
    grad_accum_steps=8,
    lr=1e-4
)

t1_val_preds_b = generate_predictions(t1_model_b, t1_tok_b, t1_val["Abstract"].tolist())
t1_metrics_b = compute_all_metrics(t1_val_preds_b, t1_val["Highlights"].tolist())
print("Task1 Model B metrics:", t1_metrics_b)

del t1_model_b, t1_train_ds_b, t1_val_ds_b
clear_memory()
gpu_mem_report("after freeing task1 model b")

Resumed model from C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task1_model_b\last_checkpoint
[before starting training for google/pegasus-xsum] allocated=3.37GB reserved=3.39GB


C:\Users\VGST\AppData\Local\Temp\ipykernel_17028\29908453.py:75: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch 1/4 bs=4 train:   0%|          | 0/2500 [00:00<?, ?it/s]

C:\Users\VGST\AppData\Local\Temp\ipykernel_17028\29908453.py:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 train_loss=nan final_batch_size=4


Epoch 1/4 val:   0%|          | 0/497 [00:00<?, ?it/s]

... (18 outputs left out)

C:\Users\VGST\AppData\Local\Temp\ipykernel_17028\29908453.py:172: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 4 val_loss=2.9096
[after loading best checkpoint from C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task1_model_b\best_checkpoint] allocated=8.11GB reserved=8.33GB


C:\Users\VGST\AppData\Local\Temp\ipykernel_17028\470968428.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


Task1 Model B metrics: {'rouge1': np.float64(0.3544485973650396), 'rouge2': np.float64(0.13438218567267082), 'rougeL': np.float64(0.2475260129996667), 'meteor': np.float64(0.3278500319418379), 'bertscore_f1': 0.8725282108153144, 'bleu': 9.947376291075352, 'chrf': 41.99460364599514, 'avg_length': 64.08413098236775}
[after freeing task1 model b] allocated=1.09GB reserved=1.33GB


In [13]:
t1_ensemble_preds_val = [
    a if t1_metrics_a["rougeL"] >= t1_metrics_b["rougeL"] else b
    for a, b in zip(t1_val_preds_a, t1_val_preds_b)
]
t1_metrics_ensemble = compute_all_metrics(t1_ensemble_preds_val, t1_val["Highlights"].tolist())
print("Task1 Ensemble metrics:", t1_metrics_ensemble)

t1_results_table = pd.DataFrame([
    {"model": T1_MODEL_A_NAME, **t1_metrics_a},
    {"model": T1_MODEL_B_NAME, **t1_metrics_b},
    {"model": "ensemble", **t1_metrics_ensemble}
])
print(t1_results_table)

T1_BEST_IS_A = t1_metrics_a["rougeL"] >= t1_metrics_b["rougeL"]

Task1 Ensemble metrics: {'rouge1': np.float64(0.3544485973650396), 'rouge2': np.float64(0.13438218567267082), 'rougeL': np.float64(0.2475260129996667), 'meteor': np.float64(0.3278500319418379), 'bertscore_f1': 0.8725282108153144, 'bleu': 9.947376291075352, 'chrf': 41.99460364599514, 'avg_length': 64.08413098236775}
                 model    rouge1    rouge2    rougeL    meteor  bertscore_f1  \
0   facebook/bart-base  0.091338  0.028112  0.063707  0.079641      0.811977   
1  google/pegasus-xsum  0.354449  0.134382  0.247526  0.327850      0.872528   
2             ensemble  0.354449  0.134382  0.247526  0.327850      0.872528   

       bleu       chrf  avg_length  
0  1.130427  12.961839   18.385894  
1  9.947376  41.994604   64.084131  
2  9.947376  41.994604   64.084131  


In [14]:
T1_MODEL_A_BEST = os.path.join(T1_MODEL_A_DIR, "best_checkpoint")
T1_MODEL_B_BEST = os.path.join(T1_MODEL_B_DIR, "best_checkpoint")

t1_test_preds_a = generate_from_checkpoint(T1_MODEL_A_BEST, t1_test["Abstract"].tolist())
t1_test_preds_b = generate_from_checkpoint(T1_MODEL_B_BEST, t1_test["Abstract"].tolist())

t1_test_preds_ensemble = t1_test_preds_a if T1_BEST_IS_A else t1_test_preds_b

filename_col = t1_test["Filename"] if "Filename" in t1_test.columns else pd.Series(range(len(t1_test)))

sub_t1_a = pd.DataFrame({"Filename": filename_col, "Abstract": t1_test["Abstract"], "Predicted_Highlights": t1_test_preds_a})
sub_t1_b = pd.DataFrame({"Filename": filename_col, "Abstract": t1_test["Abstract"], "Predicted_Highlights": t1_test_preds_b})
sub_t1_ensemble = pd.DataFrame({"Filename": filename_col, "Abstract": t1_test["Abstract"], "Predicted_Highlights": t1_test_preds_ensemble})

sub_t1_a.to_csv(os.path.join(OUTPUT_DIR, "submission_task1_model1.csv"), index=False)
sub_t1_b.to_csv(os.path.join(OUTPUT_DIR, "submission_task1_model2.csv"), index=False)
sub_t1_ensemble.to_csv(os.path.join(OUTPUT_DIR, "submission_task1_bestensemble.csv"), index=False)

print("Task 1 submissions saved")

c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\models\bart\configuration_bart.py:176: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(
C:\Users\VGST\AppData\Local\Temp\ipykernel_17028\470968428.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\transformers\generation\utils.py:1493: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


Task 1 submissions saved


Task 2

In [10]:
t2_train = load_and_clean_csv(TASK2_TRAIN, ["Abstract", "Title"])
t2_val = load_and_clean_csv(TASK2_VAL, ["Abstract", "Title"])
t2_test = load_and_clean_csv(TASK2_TEST, ["Abstract"])

print(t2_train.shape, t2_val.shape, t2_test.shape)

Loaded C:/Users/VGST/Desktop/SciHigh-2026/Task 2/SpringerSSAT-Val-SciHigh.csv using fallback encoding: cp1252
(2778, 2) (347, 2) (347, 1)


In [11]:
T2_MODEL_A_NAME = "google/flan-t5-small"
T2_MODEL_A_DIR = os.path.join(CKPT_DIR, "task2_model_a")
T2_PREFIX = "summarize the title for: "

t2_tok_a = AutoTokenizer.from_pretrained(T2_MODEL_A_NAME)
t2_train_ds_a = Seq2SeqDataset(
    [T2_PREFIX + a for a in t2_train["Abstract"]], t2_train["Title"], t2_tok_a, 512, 48
)
t2_val_ds_a = Seq2SeqDataset(
    [T2_PREFIX + a for a in t2_val["Abstract"]], t2_val["Title"], t2_tok_a, 512, 48
)

t2_model_a = train_seq2seq_model(
    T2_MODEL_A_NAME,
    t2_train_ds_a,
    t2_val_ds_a,
    t2_tok_a,
    save_dir=T2_MODEL_A_DIR,
    max_epochs=6,
    base_batch_size=4,
    grad_accum_steps=8,
    lr=5e-5
)

t2_val_input_a = [T2_PREFIX + a for a in t2_val["Abstract"]]
t2_val_preds_a = generate_predictions(t2_model_a, t2_tok_a, t2_val_input_a, max_target_len=48, num_beams=6)
t2_metrics_a = compute_all_metrics(t2_val_preds_a, t2_val["Title"].tolist())
print("Task2 Model A metrics:", t2_metrics_a)

del t2_model_a, t2_train_ds_a, t2_val_ds_a
clear_memory()
gpu_mem_report("after freeing task2 model a")

Resumed model from C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task2_model_a\last_checkpoint


c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


[before starting training for google/flan-t5-small] allocated=3.28GB reserved=3.29GB


C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\2739772654.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch 1/6 bs=4 train:   0%|          | 0/695 [00:00<?, ?it/s]

C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\2739772654.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


... (8 outputs left out)

C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\2739772654.py:177: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 2 val_loss=nan
Early stopping triggered
best_checkpoint not found in C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task2_model_a, using last_checkpoint instead
[after loading best checkpoint from C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task2_model_a\last_checkpoint] allocated=9.83GB reserved=10.18GB


C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\470968428.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Task2 Model A metrics: {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0, 'meteor': 0.0, 'bertscore_f1': 0.7530492004812279, 'bleu': 0.0, 'chrf': 1.2553072370342255, 'avg_length': 1.0}
[after freeing task2 model a] allocated=1.09GB reserved=1.20GB


In [12]:
T2_MODEL_B_NAME = "sshleifer/distilbart-cnn-6-6"
T2_MODEL_B_DIR = os.path.join(CKPT_DIR, "task2_model_b")

t2_tok_b = load_with_retry(AutoTokenizer.from_pretrained, T2_MODEL_B_NAME)
t2_train_ds_b = Seq2SeqDataset(t2_train["Abstract"], t2_train["Title"], t2_tok_b, 512, 48)
t2_val_ds_b = Seq2SeqDataset(t2_val["Abstract"], t2_val["Title"], t2_tok_b, 512, 48)

t2_model_b = train_seq2seq_model(
    T2_MODEL_B_NAME,
    t2_train_ds_b,
    t2_val_ds_b,
    t2_tok_b,
    save_dir=T2_MODEL_B_DIR,
    max_epochs=6,
    base_batch_size=4,
    grad_accum_steps=8,
    lr=3e-5
)

t2_val_preds_b = generate_predictions(t2_model_b, t2_tok_b, t2_val["Abstract"].tolist(), max_target_len=48, num_beams=6)
t2_metrics_b = compute_all_metrics(t2_val_preds_b, t2_val["Title"].tolist())
print("Task2 Model B metrics:", t2_metrics_b)

del t2_model_b, t2_train_ds_b, t2_val_ds_b
clear_memory()
gpu_mem_report("after freeing task2 model b")

Resumed model from C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task2_model_b\last_checkpoint
[before starting training for sshleifer/distilbart-cnn-6-6] allocated=2.71GB reserved=2.74GB


C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\2739772654.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch 1/6 bs=4 train:   0%|          | 0/695 [00:00<?, ?it/s]

C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\2739772654.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 train_loss=1.9596 final_batch_size=4


Epoch 1/6 val:   0%|          | 0/87 [00:00<?, ?it/s]

... (12 outputs left out)

C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\2739772654.py:177: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 3 val_loss=2.8010
Early stopping triggered
[after loading best checkpoint from C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task2_model_b\best_checkpoint] allocated=5.99GB reserved=6.09GB


C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\470968428.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


Task2 Model B metrics: {'rouge1': 0.40117464896218424, 'rouge2': 0.18915632169243612, 'rougeL': 0.33431295344635226, 'meteor': 0.3544756918675116, 'bertscore_f1': 0.888243030574205, 'bleu': 13.134449645817504, 'chrf': 41.206364728792984, 'avg_length': 11.90778097982709}
[after freeing task2 model b] allocated=1.09GB reserved=1.20GB


In [13]:
t2_ensemble_preds_val = [
    a if t2_metrics_a["rougeL"] >= t2_metrics_b["rougeL"] else b
    for a, b in zip(t2_val_preds_a, t2_val_preds_b)
]
t2_metrics_ensemble = compute_all_metrics(t2_ensemble_preds_val, t2_val["Title"].tolist())
print("Task2 Ensemble metrics:", t2_metrics_ensemble)

t2_results_table = pd.DataFrame([
    {"model": T2_MODEL_A_NAME, **t2_metrics_a},
    {"model": T2_MODEL_B_NAME, **t2_metrics_b},
    {"model": "ensemble", **t2_metrics_ensemble}
])
print(t2_results_table)

T2_BEST_IS_A = t2_metrics_a["rougeL"] >= t2_metrics_b["rougeL"]
T2_MODEL_A_BEST = resolve_checkpoint_path(T2_MODEL_A_DIR)
T2_MODEL_B_BEST = resolve_checkpoint_path(T2_MODEL_B_DIR)
T2_BEST_CHECKPOINT = T2_MODEL_A_BEST if T2_BEST_IS_A else T2_MODEL_B_BEST
T2_BEST_PREFIX = T2_PREFIX if T2_BEST_IS_A else ""

Task2 Ensemble metrics: {'rouge1': 0.40117464896218424, 'rouge2': 0.18915632169243612, 'rougeL': 0.33431295344635226, 'meteor': 0.3544756918675116, 'bertscore_f1': 0.888243030574205, 'bleu': 13.134449645817504, 'chrf': 41.206364728792984, 'avg_length': 11.90778097982709}
                          model    rouge1    rouge2    rougeL    meteor  \
0          google/flan-t5-small  0.000000  0.000000  0.000000  0.000000   
1  sshleifer/distilbart-cnn-6-6  0.401175  0.189156  0.334313  0.354476   
2                      ensemble  0.401175  0.189156  0.334313  0.354476   

   bertscore_f1      bleu       chrf  avg_length  
0      0.753049   0.00000   1.255307    1.000000  
1      0.888243  13.13445  41.206365   11.907781  
2      0.888243  13.13445  41.206365   11.907781  
best_checkpoint not found in C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task2_model_a, using last_checkpoint instead


In [14]:
t2_test_input_a = [T2_PREFIX + a for a in t2_test["Abstract"]]
t2_test_preds_a = generate_from_checkpoint(T2_MODEL_A_BEST, t2_test_input_a, max_target_len=48, num_beams=6)
t2_test_preds_b = generate_from_checkpoint(T2_MODEL_B_BEST, t2_test["Abstract"].tolist(), max_target_len=48, num_beams=6)

t2_test_preds_ensemble = t2_test_preds_a if T2_BEST_IS_A else t2_test_preds_b

sub_t2_a = pd.DataFrame({"Abstract": t2_test["Abstract"], "Predicted_Title": t2_test_preds_a})
sub_t2_b = pd.DataFrame({"Abstract": t2_test["Abstract"], "Predicted_Title": t2_test_preds_b})
sub_t2_ensemble = pd.DataFrame({"Abstract": t2_test["Abstract"], "Predicted_Title": t2_test_preds_ensemble})

sub_t2_a.to_csv(os.path.join(OUTPUT_DIR, "submission_task2_model1.csv"), index=False)
sub_t2_b.to_csv(os.path.join(OUTPUT_DIR, "submission_task2_model2.csv"), index=False)
sub_t2_ensemble.to_csv(os.path.join(OUTPUT_DIR, "submission_task2_bestensemble.csv"), index=False)

print("Task 2 submissions saved")

C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\470968428.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


Task 2 submissions saved


Task 3

In [15]:
raw_t3_train_cols = read_csv_robust(TASK3_TRAIN).columns.tolist()
raw_t3_val_cols = read_csv_robust(TASK3_VAL).columns.tolist()
raw_t3_test_cols = read_csv_robust(TASK3_TEST).columns.tolist()

print("Task 3 train columns:", raw_t3_train_cols)
print("Task 3 val columns:", raw_t3_val_cols)
print("Task 3 test columns:", raw_t3_test_cols)

ABSTRACT_CANDIDATES = ["Abstract", "abstract", "Abstracts"]
TITLE_EN_CANDIDATES = ["Title_English", "English_Title", "Title_En", "English Title", "Title_english", "Title"]
TITLE_BN_CANDIDATES = ["Title_Bengali", "Bengali_Title", "Title_Bn", "Bengali Title", "Title_bengali", "Bengali"]

t3_train_raw = read_csv_robust(TASK3_TRAIN)
T3_ABSTRACT_COL = resolve_column(t3_train_raw, ABSTRACT_CANDIDATES)
T3_TITLE_EN_COL = resolve_column(t3_train_raw, TITLE_EN_CANDIDATES)
T3_TITLE_BN_COL = resolve_column(t3_train_raw, TITLE_BN_CANDIDATES)

print("Resolved columns:", T3_ABSTRACT_COL, T3_TITLE_EN_COL, T3_TITLE_BN_COL)

t3_train = load_and_clean_csv(TASK3_TRAIN, [T3_ABSTRACT_COL, T3_TITLE_EN_COL, T3_TITLE_BN_COL])
t3_val = load_and_clean_csv(TASK3_VAL, [T3_ABSTRACT_COL, T3_TITLE_EN_COL, T3_TITLE_BN_COL])
t3_test = load_and_clean_csv(TASK3_TEST, [T3_ABSTRACT_COL])

print(t3_train.shape, t3_val.shape, t3_test.shape)

Task 3 train columns: ['Abstract', 'English_Title', 'Bengali_Title']
Task 3 val columns: ['Abstract', 'English_Title', 'Bengali_Title']
Task 3 test columns: ['Abstract', 'English_Title']
Resolved columns: Abstract English_Title Bengali_Title
(60, 3) (20, 3) (20, 2)


In [16]:
t3_val_english_preds = generate_from_checkpoint(
    T2_BEST_CHECKPOINT,
    [T2_BEST_PREFIX + a for a in t3_val[T3_ABSTRACT_COL].tolist()],
    max_target_len=48,
    num_beams=6
)

t3_english_metrics = compute_all_metrics(t3_val_english_preds, t3_val[T3_TITLE_EN_COL].tolist())
print("Task3 English title metrics:", t3_english_metrics)

C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\470968428.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


Task3 English title metrics: {'rouge1': 0.9666004634227077, 'rouge2': 0.9403743315508022, 'rougeL': 0.9576458036984354, 'meteor': 0.9575507463246516, 'bertscore_f1': 0.9914031445980072, 'bleu': 93.05274252352265, 'chrf': 96.00781186787218, 'avg_length': 14.4}


In [17]:
T3_MODEL_A_NAME = "facebook/nllb-200-distilled-600M"
T3_MODEL_A_DIR = os.path.join(CKPT_DIR, "task3_model_a")

t3_tok_a = load_with_retry(AutoTokenizer.from_pretrained, T3_MODEL_A_NAME, src_lang="eng_Latn", tgt_lang="ben_Beng")
t3_train_ds_a = Seq2SeqDataset(t3_train[T3_TITLE_EN_COL], t3_train[T3_TITLE_BN_COL], t3_tok_a, 128, 64)
t3_val_ds_a = Seq2SeqDataset(t3_val[T3_TITLE_EN_COL], t3_val[T3_TITLE_BN_COL], t3_tok_a, 128, 64)

t3_model_a = train_seq2seq_model(
    T3_MODEL_A_NAME,
    t3_train_ds_a,
    t3_val_ds_a,
    t3_tok_a,
    save_dir=T3_MODEL_A_DIR,
    max_epochs=10,
    base_batch_size=4,
    grad_accum_steps=4,
    lr=2e-5,
    patience=3
)

try:
    nllb_forced_bos = t3_tok_a.convert_tokens_to_ids("ben_Beng")
except Exception:
    nllb_forced_bos = t3_tok_a.lang_code_to_id["ben_Beng"]

t3_val_bn_preds_a = generate_predictions(
    t3_model_a, t3_tok_a, t3_val[T3_TITLE_EN_COL].tolist(),
    max_target_len=64, num_beams=5, forced_bos_token_id=nllb_forced_bos
)
t3_metrics_a = compute_all_metrics(t3_val_bn_preds_a, t3_val[T3_TITLE_BN_COL].tolist(), lang="bn")
t3_comet_a = compute_comet(t3_val[T3_TITLE_EN_COL].tolist(), t3_val_bn_preds_a, t3_val[T3_TITLE_BN_COL].tolist())
t3_metrics_a["comet"] = t3_comet_a
print("Task3 NLLB metrics:", t3_metrics_a)

del t3_model_a, t3_train_ds_a, t3_val_ds_a
clear_memory()
gpu_mem_report("after freeing task3 model a")

Resumed model from C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task3_model_a\last_checkpoint
[before starting training for facebook/nllb-200-distilled-600M] allocated=3.56GB reserved=3.58GB


C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\2739772654.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch 1/10 bs=4 train:   0%|          | 0/15 [00:00<?, ?it/s]

C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\2739772654.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


OOM at step 6, consecutive_oom=1, total_this_epoch=1, current batch size=4
OOM at step 10, consecutive_oom=1, total_this_epoch=2, current batch size=4
OOM at step 13, consecutive_oom=1, total_this_epoch=3, current batch size=4
Epoch 1 train_loss=2.1508 final_batch_size=4


Epoch 1/10 val:   0%|          | 0/5 [00:00<?, ?it/s]

... (56 outputs left out)

C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\470968428.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():
c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\torchmetrics\utilities\imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\VGST\.conda\envs\torch_env\lib\site-packages\lightning_fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
INFO:pytorch_lightning.utilities.migration.util

Task3 NLLB metrics: {'rouge1': 0.125, 'rouge2': 0.1, 'rougeL': 0.125, 'meteor': 0.5644039076390027, 'bertscore_f1': 0.868386709690094, 'bleu': 31.012548961085926, 'chrf': 59.72031295596756, 'avg_length': 13.2, 'comet': 0.7888677716255188}
[after freeing task3 model a] allocated=0.63GB reserved=2.12GB


In [18]:
T3_MODEL_B_NAME = "facebook/m2m100_418M"
T3_MODEL_B_DIR = os.path.join(CKPT_DIR, "task3_model_b")

t3_tok_b = load_with_retry(AutoTokenizer.from_pretrained, T3_MODEL_B_NAME)
t3_tok_b.src_lang = "en"
t3_tok_b.tgt_lang = "bn"

t3_train_ds_b = Seq2SeqDataset(t3_train[T3_TITLE_EN_COL], t3_train[T3_TITLE_BN_COL], t3_tok_b, 128, 64)
t3_val_ds_b = Seq2SeqDataset(t3_val[T3_TITLE_EN_COL], t3_val[T3_TITLE_BN_COL], t3_tok_b, 128, 64)

t3_model_b = train_seq2seq_model(
    T3_MODEL_B_NAME,
    t3_train_ds_b,
    t3_val_ds_b,
    t3_tok_b,
    save_dir=T3_MODEL_B_DIR,
    max_epochs=10,
    base_batch_size=4,
    grad_accum_steps=4,
    lr=2e-5,
    patience=3
)

m2m_forced_bos = t3_tok_b.get_lang_id("bn")

t3_val_bn_preds_b = generate_predictions(
    t3_model_b, t3_tok_b, t3_val[T3_TITLE_EN_COL].tolist(),
    max_target_len=64, num_beams=5, forced_bos_token_id=m2m_forced_bos
)
t3_metrics_b = compute_all_metrics(t3_val_bn_preds_b, t3_val[T3_TITLE_BN_COL].tolist(), lang="bn")
t3_comet_b = compute_comet(t3_val[T3_TITLE_EN_COL].tolist(), t3_val_bn_preds_b, t3_val[T3_TITLE_BN_COL].tolist())
t3_metrics_b["comet"] = t3_comet_b
print("Task3 M2M100 metrics:", t3_metrics_b)

del t3_model_b, t3_train_ds_b, t3_val_ds_b
clear_memory()
gpu_mem_report("after freeing task3 model b")

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

[before starting training for facebook/m2m100_418M] allocated=2.57GB reserved=2.65GB


C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\2739772654.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


... (49 outputs left out)

C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\2739772654.py:177: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 8 val_loss=1.9105
Early stopping triggered
[after loading best checkpoint from C:\Users\VGST\Desktop\SciHigh-2026\outputs\checkpoints\task3_model_b\best_checkpoint] allocated=6.56GB reserved=6.62GB


C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\470968428.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predi

Task3 M2M100 metrics: {'rouge1': 0.125, 'rouge2': 0.1, 'rougeL': 0.125, 'meteor': 0.2719797811023568, 'bertscore_f1': 0.7882966816425323, 'bleu': 4.414331513618592, 'chrf': 36.58005391911352, 'avg_length': 12.2, 'comet': 0.6711797147989274}
[after freeing task3 model b] allocated=0.63GB reserved=2.12GB


In [19]:
t3_ensemble_preds_val = [
    a if t3_metrics_a["rougeL"] >= t3_metrics_b["rougeL"] else b
    for a, b in zip(t3_val_bn_preds_a, t3_val_bn_preds_b)
]
t3_metrics_ensemble = compute_all_metrics(t3_ensemble_preds_val, t3_val[T3_TITLE_BN_COL].tolist(), lang="bn")
t3_metrics_ensemble["comet"] = compute_comet(t3_val[T3_TITLE_EN_COL].tolist(), t3_ensemble_preds_val, t3_val[T3_TITLE_BN_COL].tolist())
print("Task3 Ensemble metrics:", t3_metrics_ensemble)

t3_results_table = pd.DataFrame([
    {"model": T3_MODEL_A_NAME, **t3_metrics_a},
    {"model": T3_MODEL_B_NAME, **t3_metrics_b},
    {"model": "ensemble", **t3_metrics_ensemble}
])
print(t3_results_table)

T3_BEST_IS_A = t3_metrics_a["rougeL"] >= t3_metrics_b["rougeL"]
T3_MODEL_A_BEST = resolve_checkpoint_path(T3_MODEL_A_DIR)
T3_MODEL_B_BEST = resolve_checkpoint_path(T3_MODEL_B_DIR)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00,  5.23it/s]


Task3 Ensemble metrics: {'rouge1': 0.125, 'rouge2': 0.1, 'rougeL': 0.125, 'meteor': 0.5644039076390027, 'bertscore_f1': 0.868386709690094, 'bleu': 31.012548961085926, 'chrf': 59.72031295596756, 'avg_length': 13.2, 'comet': 0.7888677716255188}
                              model  rouge1  rouge2  rougeL    meteor  \
0  facebook/nllb-200-distilled-600M   0.125     0.1   0.125  0.564404   
1              facebook/m2m100_418M   0.125     0.1   0.125  0.271980   
2                          ensemble   0.125     0.1   0.125  0.564404   

   bertscore_f1       bleu       chrf  avg_length     comet  
0      0.868387  31.012549  59.720313        13.2  0.788868  
1      0.788297   4.414332  36.580054        12.2  0.671180  
2      0.868387  31.012549  59.720313        13.2  0.788868  


In [ ]:
t3_test_english = generate_from_checkpoint(
    T2_BEST_CHECKPOINT,
    [T2_BEST_PREFIX + a for a in t3_test[T3_ABSTRACT_COL].tolist()],
    max_target_len=48,
    num_beams=6
)

nllb_tok_check = AutoTokenizer.from_pretrained(T3_MODEL_A_BEST, local_files_only=True)
try:
    nllb_forced_bos = nllb_tok_check.convert_tokens_to_ids("ben_Beng")
except Exception:
    nllb_forced_bos = nllb_tok_check.lang_code_to_id["ben_Beng"]
del nllb_tok_check

t3_test_bn_a = generate_from_checkpoint(
    T3_MODEL_A_BEST, t3_test_english, max_target_len=64, num_beams=5, forced_bos_token_id=nllb_forced_bos
)

m2m_tok_check = AutoTokenizer.from_pretrained(T3_MODEL_B_BEST, local_files_only=True)
m2m_tok_check.src_lang = "en"
m2m_forced_bos = m2m_tok_check.get_lang_id("bn")
del m2m_tok_check

t3_test_bn_b = generate_from_checkpoint(
    T3_MODEL_B_BEST, t3_test_english, max_target_len=64, num_beams=5, forced_bos_token_id=m2m_forced_bos
)

t3_test_bn_ensemble = t3_test_bn_a if T3_BEST_IS_A else t3_test_bn_b

sub_t3_a = pd.DataFrame({
    "Abstract": t3_test[T3_ABSTRACT_COL],
    "Predicted_Title_English": t3_test_english,
    "Predicted_Title_Bengali": t3_test_bn_a
})
sub_t3_b = pd.DataFrame({
    "Abstract": t3_test[T3_ABSTRACT_COL],
    "Predicted_Title_English": t3_test_english,
    "Predicted_Title_Bengali": t3_test_bn_b
})
sub_t3_ensemble = pd.DataFrame({
    "Abstract": t3_test[T3_ABSTRACT_COL],
    "Predicted_Title_English": t3_test_english,
    "Predicted_Title_Bengali": t3_test_bn_ensemble
})

sub_t3_a.to_csv(os.path.join(OUTPUT_DIR, "submission_task3_model1.csv"), index=False)
sub_t3_b.to_csv(os.path.join(OUTPUT_DIR, "submission_task3_model2.csv"), index=False)
sub_t3_ensemble.to_csv(os.path.join(OUTPUT_DIR, "submission_task3_bestensemble.csv"), index=False)

print("Task 3 submissions saved")

C:\Users\VGST\AppData\Local\Temp\ipykernel_17744\470968428.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


Task 3 submissions saved


: 